# Semana 12 · ¿Qué se conserva cuando todo se mueve?

Este cuaderno cierra la fase 1: simulas un péndulo con tres métodos numéricos y compruebas cuál conserva la energía. Es el ensayo de lo que harás con las órbitas en la fase 2.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. Energía cinética y potencial

Por unidad de masa, la **energía cinética** de un cuerpo que se mueve con velocidad $v$ es $\tfrac{1}{2}v^2$, y la **energía potencial gravitatoria** cerca de la superficie a una altura $h$ es $g\,h$. La suma, la **energía mecánica**, se conserva si solo actúa la gravedad. Una piedra que cae desde 20 m da la misma velocidad que con la cinemática de la semana 9:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

g = 9.80665
h = 20.0
v = np.sqrt(2 * g * h)            # toda la energía potencial g·h se convierte en cinética ½v²
print(f"v = {v:.2f} m/s  (con la cinemática: {g * np.sqrt(2 * h / g):.2f} m/s)")

## 2. El péndulo

Un péndulo de longitud $L$ con una masa en el extremo y un ángulo $\theta$ respecto a la vertical cumple

$$\ddot\theta = -\frac{g}{L}\sin\theta,$$

donde $\ddot\theta$ es la aceleración angular (la derivada de la derivada del ángulo). Por unidad de masa, su energía es

$$E = \tfrac{1}{2}L^2\dot\theta^2 + g\,L\,(1 - \cos\theta).$$

Si el péndulo se suelta desde $\theta_0$ con velocidad cero, la energía conservada es $g\,L\,(1-\cos\theta_0)$ y la velocidad máxima, abajo, vale $v = \sqrt{2gL(1-\cos\theta_0)}$.

In [ ]:
L = 1.0
theta0 = np.radians(30)
v_max = np.sqrt(2 * g * L * (1 - np.cos(theta0)))
print(f"Velocidad máxima de un péndulo de 1 m soltado desde 30°: {v_max:.3f} m/s")
print(f"Periodo para oscilaciones pequeñas 2π√(L/g): {2 * np.pi * np.sqrt(L / g):.3f} s")

## 3. Tres métodos para simularlo

Avanzamos el reloj $\Delta t$ cada vez, como en las semanas 8 y 9. La cuestión es **en qué orden** se actualizan la velocidad y la posición:

- **Euler explícito:** se actualizan ambas con los valores del paso anterior.
- **Euler-Cromer (o semiimplícito):** se actualiza primero la velocidad, y la posición con la velocidad ya nueva.
- **Velocity Verlet:** usa la aceleración al principio y al final del paso.

In [ ]:
def aceleracion(theta):
    return -(g / L) * np.sin(theta)

def energia(theta, omega):
    return 0.5 * L**2 * omega**2 + g * L * (1 - np.cos(theta))

def simula(metodo, theta0, t_final, dt):
    n = int(round(t_final / dt))
    t = np.arange(n + 1) * dt
    th, om = np.zeros(n + 1), np.zeros(n + 1)
    th[0] = theta0
    for i in range(n):
        if metodo == "euler":
            th[i + 1] = th[i] + om[i] * dt
            om[i + 1] = om[i] + aceleracion(th[i]) * dt
        elif metodo == "cromer":
            om[i + 1] = om[i] + aceleracion(th[i]) * dt
            th[i + 1] = th[i] + om[i + 1] * dt
        elif metodo == "verlet":
            a0 = aceleracion(th[i])
            th[i + 1] = th[i] + om[i] * dt + 0.5 * a0 * dt**2
            om[i + 1] = om[i] + 0.5 * (a0 + aceleracion(th[i + 1])) * dt
    return t, th, om

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for metodo in ["euler", "cromer", "verlet"]:
    t, th, om = simula(metodo, np.radians(30), 20, 0.01)
    E = energia(th, om)
    axs[0].plot(t, np.degrees(th), label=metodo, lw=1)
    axs[1].plot(t, E / E[0], label=metodo)
    print(f"{metodo:7s}  energía final / inicial = {E[-1] / E[0]:.4f}")
axs[0].set_xlabel("t (s)"); axs[0].set_ylabel("ángulo (°)"); axs[0].legend(); axs[0].grid(alpha=0.3)
axs[1].set_xlabel("t (s)"); axs[1].set_ylabel("E / E inicial"); axs[1].legend(); axs[1].grid(alpha=0.3)
axs[1].set_title("Conservación de la energía")
plt.tight_layout(); plt.show()

**Qué ver.** El método de Euler explícito **gana energía** a cada oscilación: la amplitud del péndulo crece, algo físicamente imposible. Euler-Cromer y Verlet mantienen la energía casi constante, con pequeñas oscilaciones. Para órbitas, esa diferencia es decisiva: un método que gana energía haría que un planeta se alejase poco a poco de su órbita. Por eso en la fase 2 usarás los métodos que conservan la energía.

## 4. El periodo

Para oscilaciones pequeñas el periodo es $T = 2\pi\sqrt{L/g}$, independiente de la amplitud. Pero si la amplitud es grande, el periodo crece. Lo medimos en la simulación.

In [ ]:
def periodo(theta0_grados):
    t, th, om = simula("verlet", np.radians(theta0_grados), 20, 0.001)
    cruces = np.where((th[:-1] > 0) & (th[1:] <= 0))[0]       # pasos por cero hacia valores negativos
    return np.mean(np.diff(t[cruces])) if len(cruces) > 1 else np.nan

T0 = 2 * np.pi * np.sqrt(L / g)
for amp in [5, 30, 60, 90, 120]:
    print(f"amplitud {amp:3d}°: T = {periodo(amp):.3f} s   (pequeñas oscilaciones: {T0:.3f} s)")

## 5. Para pensar

1. ¿Qué ocurre con la energía de Euler explícito si reduces `dt` a 0,001? ¿Desaparece el problema?
2. ¿Cómo cambia el periodo si alargas el péndulo al doble? Comprueba que $T \propto \sqrt{L}$.
3. Si añadieses rozamiento, ¿qué pasaría con la energía mecánica? ¿Dónde estaría la energía que falta?